In [ ]:
import numpy as np
import seaborn
import matplotlib.pyplot as plt
import pandas as pd

# Import data

In [ ]:
def import_data():
    with open("phishing.data", 'r') as f:
        data = f.read().splitlines()
    data = [list(map(int, line.split(','))) for line in data]
    X = np.array([[line[5],line[21],line[25]] for line in data])
    y = np.array([line[-1] for line in data])
    return X, y

In [ ]:
# gauss Kernel
def Kernel(x1, x2, sigma=1):
        if np.ndim(x1) == 1 and np.ndim(x2) == 1:
            return np.exp(-(np.linalg.norm(x1-x2,2))**2/(2*sigma**2))
        elif(np.ndim(x1)>1 and np.ndim(x2) == 1) or (np.ndim(x1) == 1 and np.ndim(x2)>1):
            return np.exp(-(np.linalg.norm(x1-x2, 2, axis=1)**2)/(2*sigma**2))
        elif np.ndim(x1) > 1 and np.ndim(x2) > 1 :
            return np.exp(-(np.linalg.norm(x1[:, np.newaxis] \
                             - x2[np.newaxis, :], 2, axis = 2) ** 2)/(2*sigma**2))
        return 0.

In [ ]:
def get_gram_matrix(X, sigma=1.0):
    sq_norms = np.sum(X**2, axis=1).reshape(-1, 1)
    

    dists_sq = sq_norms + sq_norms.T - 2 * np.dot(X, X.T)
    dists_sq = np.maximum(dists_sq, 0)
    
    return np.exp(-dists_sq / (2 * sigma**2))

In [ ]:
# linear Kernel
def Kernel(x1, x2):
        return x1 @ x2.T

In [ ]:
# linear Kernel
def get_gram_matrix(X):
    return X @ X.T

# utils

In [ ]:
def error(i, X, y, alphas, b,K):
    f_xi = np.dot(alphas * y, K[i]) + b
    return f_xi - y[i]

In [ ]:
def result(X_test, X_train, y_train, alphas, b,sigma=1.0):
    X_test = np.asarray(X_test)
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)
    
    y_scores = []
    for x in X_test:
        x = x.flatten()
        score = 0.0
        for i in range(len(X_train)):
            xi = X_train[i].flatten()
            score += alphas[i] * y_train[i] * Kernel(xi, x, sigma)
        score += b
        y_scores.append(score)
    return np.array(y_scores)


In [ ]:
# only for linear kernel
def result(X_test, X_train, y_train, alphas, b):
    K_test = X_test @ X_train.T  
    scores = (alphas * y_train) @ K_test.T + b
    return scores

# single iteration

In [ ]:
def step(i,j,X,y,alphas,b,C,K):
    if i==j: return False, b


    y_i=y[i]
    y_j=y[j]

    alpha_i=alphas[i]
    alpha_j=alphas[j]

    E_i = error(i,X,y,alphas,b,K)
    E_j = error(j,X,y,alphas,b,K)

    if y_i != y_j:
        L = max(0, alpha_j - alpha_i)
        H = min(C, C + alpha_j - alpha_i)
    else:
        L = max(0, alpha_i + alpha_j - C)
        H = min(C, alpha_i + alpha_j)
    if L == H: return False, b

    k_ii = K[i, i]
    k_jj = K[j, j]
    k_ij = K[i, j]

    eta = 2 * k_ij - k_ii - k_jj
    if eta >= 0: return False, b
    alpha_j_new = alpha_j - (y_j * (E_i - E_j)) / eta
    if alpha_j_new > H:
        alpha_j_new = H
    elif alpha_j_new < L:
        alpha_j_new = L

    # small changes, skip this pair
    if abs(alpha_j_new - alpha_j) < 1e-5: return False, b

    alpha_i_new = alpha_i + y_i * y_j * (alpha_j - alpha_j_new)


    # 2 potential new b's
    b_i = b - E_i - y_i * (alpha_i_new - alpha_i) * k_ii - y_j * (alpha_j_new - alpha_j) * k_ij
    b_j = b - E_j - y_i * (alpha_i_new - alpha_i) * k_ij - y_j * (alpha_j_new - alpha_j) * k_jj

    if 0 < alpha_i_new < C:
        b_new = b_i
    elif 0 < alpha_j_new < C:
        b_new = b_j
    else:
        b_new = (b_i + b_j) / 2

    alphas[i] = alpha_i_new
    alphas[j] = alpha_j_new

    return True, b_new


# outer loop

In [ ]:
def SMO(X, y, n, C = 1.0):
    K = get_gram_matrix(X)
    alphas = np.zeros(n)
    b = 0
    max_passes = 5
    max_iterations = 150
    iteration = 0
    passes = 0

    while passes < max_passes and iteration < max_iterations:
        iteration += 1
        if iteration % 10 == 0:
            print(f"Iteracja {iteration}/{max_iterations}... Pass {passes}/{max_passes}...")
        num_changed_alphas = 0
        for i in range(n):
            if i % 1000 == 0:
                print(f"Przetwarzanie punktu {i}/{n}...",end="\r")
            j = np.random.choice([x for x in range(n) if x != i])
            result,b_new = step(i,j,X,y,alphas,b,C, K)
            if result == True:
                b = b_new
                num_changed_alphas += 1
        if num_changed_alphas == 0:
            passes += 1
            print(f"Pass {passes}/{max_passes}... Brak zmian w alphas.")
        else:
            passes = 0
    return alphas, b

# Test results

In [ ]:
X, y = import_data()
X_train_orig = []
X_test_orig = []
X_val_orig = []
y_train_orig = []
y_test_orig = []
y_val_orig = []

for i in range(len(X)):
    num = np.random.rand()
    if num < 0.6:
        X_train_orig.append(X[i])
        y_train_orig.append(y[i])
    elif num < 0.8:
        X_val_orig.append(X[i])
        y_val_orig.append(y[i])
    else:
        X_test_orig.append(X[i])
        y_test_orig.append(y[i])
C_params = [10]
best_accuracy = 0
best_C = None
best_alphas = None
best_b = None
for C in C_params:
    X_train = np.array(X_train_orig)
    y_train = np.array(y_train_orig)
    X_val = np.array(X_val_orig)
    y_val = np.array(y_val_orig)
    X_test = np.array(X_test_orig)
    y_test = np.array(y_test_orig)

    alphas, b = SMO(X_train, y_train, len(X_train),C)
    y_pred = result(X_test, X_train, y_train, alphas, b)
    y_pred = np.where(y_pred >= 0, 1, -1)
    accuracy = np.mean(y_pred == y_test)
    print(f"for C={C}: Accuracy: {accuracy*100:.2f}%")
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_C = C
        best_alphas = alphas
        best_b = b
y_pred = result(X_test, X_train, y_train, best_alphas, best_b)
y_pred = np.where(y_pred >= 0, 1, -1)
print(f"Best C: {best_C}, Best Accuracy: {best_accuracy*100:.2f}%")

def get_confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == -1) & (y_pred == -1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    return tp, tn, fp, fn

tp, tn, fp, fn = get_confusion_matrix(y_test, y_pred)
print(f"test size: {len(y_test)}")
print("Confusion Matrix:")
print(f"TP: {tp} | FP: {fp}")
print(f"FN: {fn} | TN: {tn}")

In [ ]:
# find best sigma for RBF kernel
X, y = import_data()
X_train_orig = []
X_test_orig = []
X_val_orig = []
y_train_orig = []
y_test_orig = []
y_val_orig = []

for i in range(len(X)):
    num = np.random.rand()
    if num < 0.6:
        X_train_orig.append(X[i])
        y_train_orig.append(y[i])
    elif num < 0.8:
        X_val_orig.append(X[i])
        y_val_orig.append(y[i])
    else:
        X_test_orig.append(X[i])
        y_test_orig.append(y[i])
sigmas = [0.1, 0.5, 1, 5, 10]
best_accuracy = 0
C=10
best_alphas = None
best_b = None
best_sigma = None
for sigma in sigmas:
    X_train = np.array(X_train_orig)
    y_train = np.array(y_train_orig)
    X_val = np.array(X_val_orig)
    y_val = np.array(y_val_orig)
    X_test = np.array(X_test_orig)
    y_test = np.array(y_test_orig)

    alphas, b = SMO(X_train, y_train, len(X_train),C)
    y_pred = result(X_test, X_train, y_train, alphas, b,sigma)
    y_pred = np.where(y_pred >= 0, 1, -1)
    accuracy = np.mean(y_pred == y_test)
    print(f"for sigma={sigma}: Accuracy: {accuracy*100:.2f}%")
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_alphas = alphas
        best_b = b
        best_sigma = sigma
y_pred = result(X_test, X_train, y_train, best_alphas, best_b, best_sigma)
y_pred = np.where(y_pred >= 0, 1, -1)
print(f"Best sigma: {best_sigma}, Best Accuracy: {best_accuracy*100:.2f}%")

def get_confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == -1) & (y_pred == -1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    return tp, tn, fp, fn

tp, tn, fp, fn = get_confusion_matrix(y_test, y_pred)
print(f"test size: {len(y_test)}")
print("Confusion Matrix:")
print(f"TP: {tp} | FP: {fp}")
print(f"FN: {fn} | TN: {tn}")